# Prepare Hourly Data

Add two half-hour readings to make one hourly value.

- **Part 1** runs the steps on the four-home sample (`data/pilot/half_hour_sample.csv`).
- **Part 2** applies the same rules to all 300 selected homes and saves `data/processed/hourly_readings.csv`.
- **Part 3** checks the file with its users (M3 and M5) and explains each reported value against the raw readings.

Rules: raw `tstp` values mark half-hour **ends** in UTC. Subtract 30 minutes, then group by UTC hour.
An hour has a total only when **both** of its half-hour readings are valid. Otherwise `kWh` stays missing (never 0).
A measured 0.000 reading is a valid reading.

Open this notebook in `notebooks/M2/`. Run the cells in order. Saving again replaces this notebook's output files.

# Part 1: Four-home sample

## Choose the project folder

In [1]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('C:/Projects/household-energy-prediction')

## Read the sample

half-hour consumption readings for 4 homes

In [2]:
import pandas as pd
raw_path = PROJECT_ROOT / "data/pilot/half_hour_sample.csv"
raw = pd.read_csv(raw_path, dtype=str)
raw.head()

,LCLid,tstp,energy(kWh/hh)
0,MAC000414,2013-01-01 00:00:00.0000000,0.13
1,MAC000414,2013-01-01 00:30:00.0000000,0.125
2,MAC000414,2013-01-01 01:00:00.0000000,0.148
3,MAC000414,2013-01-01 01:30:00.0000000,0.095
4,MAC000414,2013-01-01 02:00:00.0000000,0.098


In [3]:
raw['LCLid'].unique() # 4 homes

array(['MAC000414', 'MAC000675', 'MAC001312', 'MAC005456'], dtype=object)

## Keep the four homes

In [4]:
homes = ["MAC000414", "MAC000675", "MAC001312", "MAC005456"]
raw = raw[raw["LCLid"].isin(homes)].copy()
raw["LCLid"].value_counts()

LCLid
MAC001312    17520
MAC000675    17517
MAC000414    17325
MAC005456    16894
Name: count, dtype: int64

## Read dates and energy

In [5]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69256 entries, 0 to 69255
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LCLid           69256 non-null  object
 1   tstp            69256 non-null  object
 2   energy(kWh/hh)  69256 non-null  object
dtypes: object(3)
memory usage: 1.6+ MB


In [6]:
parsed = raw[["LCLid"]].copy()
parsed["ts"] = pd.to_datetime(raw["tstp"], utc=True) # convert dates to "datetime" datatype
parsed["energy"] = pd.to_numeric(raw["energy(kWh/hh)"].str.strip(), errors="coerce") # convert energy to numeric datatype
print(parsed.info())
parsed.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69256 entries, 0 to 69255
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype              
---  ------  --------------  -----              
 0   LCLid   69256 non-null  object             
 1   ts      69256 non-null  datetime64[ns, UTC]
 2   energy  69256 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(1), object(1)
memory usage: 1.6+ MB
None


,LCLid,ts,energy
0,MAC000414,2013-01-01 00:00:00+00:00,0.130
1,MAC000414,2013-01-01 00:30:00+00:00,0.125
2,MAC000414,2013-01-01 01:00:00+00:00,0.148
3,MAC000414,2013-01-01 01:30:00+00:00,0.095
4,MAC000414,2013-01-01 02:00:00+00:00,0.098


## Find the start of each half-hour

Raw times mark half-hour ends. Subtract 30 minutes before grouping.

In [7]:
interval_start = parsed["ts"] - pd.Timedelta(minutes=30)
parsed["hour_start"] = interval_start.dt.floor("h")
parsed.head()

,LCLid,ts,energy,hour_start
0,MAC000414,2013-01-01 00:00:00+00:00,0.130,2012-12-31 23:00:00+00:00
1,MAC000414,2013-01-01 00:30:00+00:00,0.125,2013-01-01 00:00:00+00:00
2,MAC000414,2013-01-01 01:00:00+00:00,0.148,2013-01-01 00:00:00+00:00
3,MAC000414,2013-01-01 01:30:00+00:00,0.095,2013-01-01 01:00:00+00:00
4,MAC000414,2013-01-01 02:00:00+00:00,0.098,2013-01-01 01:00:00+00:00


## Add readings for each hour

Keep the sum when both readings are present. Otherwise, leave it missing.

In [8]:
hourly = parsed.groupby(["LCLid", "hour_start"])["energy"].agg(n_valid="count", kWh_sum="sum")
hourly["kWh"] = hourly["kWh_sum"].where(hourly["n_valid"] == 2).round(3)
hourly.head()

n_valid  kWh_sum    kWh
LCLid     hour_start                                        
MAC000414 2012-12-31 23:00:00+00:00        1    0.130    NaN
          2013-01-01 00:00:00+00:00        2    0.273  0.273
          2013-01-01 01:00:00+00:00        2    0.193  0.193
          2013-01-01 02:00:00+00:00        2    0.090  0.090
          2013-01-01 03:00:00+00:00        2    0.091  0.091

## List all hours in 2013

In [9]:
hours = pd.date_range("2013-01-01 00:00", "2013-12-31 23:00", freq="h", tz="UTC")
grid = pd.MultiIndex.from_product([sorted(homes), hours], names=["LCLid", "hour_start"])
len(grid)

35040

## Keep missing hours in the table

In [10]:
hourly = hourly.reindex(grid)
hourly["n_valid"] = hourly["n_valid"].fillna(0).astype(int)
hourly["is_complete"] = hourly["n_valid"] == 2
hourly = hourly[["kWh", "n_valid", "is_complete"]].reset_index()
hourly.head()

,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,2,True
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,2,True
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,2,True
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,2,True
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,2,True


## Count complete and missing hours

In [11]:
hourly["is_complete"].value_counts()

is_complete
True     34620
False      420
Name: count, dtype: int64

## Choose one hour to check

In [12]:
example_home = "MAC001312"
example_hour = pd.Timestamp("2013-01-01 00:00", tz="UTC")
example_end = example_hour + pd.Timedelta(hours=1)
example_home, example_hour

('MAC001312', Timestamp('2013-01-01 00:00:00+0000', tz='UTC'))

## Show its two half-hour readings

In [13]:
home_rows = parsed[parsed["LCLid"] == example_home]
raw_example = home_rows[
    (home_rows["ts"] > example_hour) & (home_rows["ts"] <= example_end)
]
raw_example[["LCLid", "ts", "energy"]]

,LCLid,ts,energy
34843,MAC001312,2013-01-01 00:30:00+00:00,0.018
34844,MAC001312,2013-01-01 01:00:00+00:00,0.041


## Show the saved hourly value

In [14]:
hourly_example = hourly[
    (hourly["LCLid"] == example_home) & (hourly["hour_start"] == example_hour)
]
hourly_example

,LCLid,hour_start,kWh,n_valid,is_complete
17520,MAC001312,2013-01-01 00:00:00+00:00,0.059,2,True


## Compare the sum

In [15]:
manual_sum = round(raw_example["energy"].sum(), 3)
saved_sum = hourly_example["kWh"].iloc[0]
manual_sum, saved_sum

(0.059, 0.059)

## Choose the output folder

In [16]:
output_dir = PROJECT_ROOT / "data/pilot"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "hourly_sample.csv"
output_path

WindowsPath('C:/Projects/household-energy-prediction/data/pilot/hourly_sample.csv')

## Save the hourly file

In [17]:
hourly.to_csv(output_path, index=False)
hourly.head()

,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000414,2013-01-01 00:00:00+00:00,0.273,2,True
1,MAC000414,2013-01-01 01:00:00+00:00,0.193,2,True
2,MAC000414,2013-01-01 02:00:00+00:00,0.090,2,True
3,MAC000414,2013-01-01 03:00:00+00:00,0.091,2,True
4,MAC000414,2013-01-01 04:00:00+00:00,0.127,2,True


# Part 2: All 300 selected homes (M2-W1)

The same steps as Part 1, applied to the full raw file.

## Read the selected home IDs

In [18]:
ids_path = PROJECT_ROOT / "data/raw/selected_household_ids.csv"
selected_ids = pd.read_csv(ids_path, dtype=str)
selected_homes = sorted(selected_ids["LCLid"].unique())
len(selected_ids), len(selected_homes)

(300, 300)

## Read the full half-hour file

In [19]:
full_raw_path = PROJECT_ROOT / "data/raw/half_hour_readings_2013.csv"
full_raw = pd.read_csv(full_raw_path, dtype=str)
print("raw rows:", len(full_raw))
print("homes in raw file:", full_raw["LCLid"].nunique())
print("raw file homes match the selected list:", set(full_raw["LCLid"]) == set(selected_homes))
full_raw.head()

raw rows: 5219816
homes in raw file: 300
raw file homes match the selected list: True


,LCLid,tstp,energy(kWh/hh)
0,MAC003737,2013-01-01 00:00:00.0000000,0.088
1,MAC003737,2013-01-01 00:30:00.0000000,0.109
2,MAC003737,2013-01-01 01:00:00.0000000,0.075
3,MAC003737,2013-01-01 01:30:00.0000000,0.076
4,MAC003737,2013-01-01 02:00:00.0000000,0.108


## Keep the selected homes

In [20]:
full_raw = full_raw[full_raw["LCLid"].isin(selected_homes)].copy()
full_raw["LCLid"].nunique(), len(full_raw)

(300, 5219816)

## Read dates and energy

In [21]:
full = full_raw[["LCLid"]].copy()
full["ts"] = pd.to_datetime(full_raw["tstp"], utc=True)
full["energy"] = pd.to_numeric(full_raw["energy(kWh/hh)"].str.strip(), errors="coerce")
full.head()

,LCLid,ts,energy
0,MAC003737,2013-01-01 00:00:00+00:00,0.088
1,MAC003737,2013-01-01 00:30:00+00:00,0.109
2,MAC003737,2013-01-01 01:00:00+00:00,0.075
3,MAC003737,2013-01-01 01:30:00+00:00,0.076
4,MAC003737,2013-01-01 02:00:00+00:00,0.108


## Check the raw readings

Duplicates or times off the half-hour grid would give wrong hourly sums, so the notebook stops if any appear.

In [22]:
raw_checks = {
    "unparsed_times": int(full["ts"].isna().sum()),
    "non_numeric_energy": int(full["energy"].isna().sum()),
    "duplicate_home_time": int(full.duplicated(["LCLid", "ts"]).sum()),
    "off_grid_times": int((~full["ts"].dt.minute.isin([0, 30]) | (full["ts"].dt.second != 0)).sum()),
    "measured_zero_readings": int((full["energy"] == 0).sum()),
    "first_raw_time": full["ts"].min(),
    "last_raw_time": full["ts"].max(),
}
assert raw_checks["unparsed_times"] == 0
assert raw_checks["duplicate_home_time"] == 0
assert raw_checks["off_grid_times"] == 0
raw_checks

{'unparsed_times': 0,
 'non_numeric_energy': 0,
 'duplicate_home_time': 0,
 'off_grid_times': 0,
 'measured_zero_readings': 70005,
 'first_raw_time': Timestamp('2013-01-01 00:00:00+0000', tz='UTC'),
 'last_raw_time': Timestamp('2013-12-31 23:30:00+0000', tz='UTC')}

## Find the start of each half-hour

Raw times mark half-hour ends. Subtract 30 minutes before grouping.
The reading at 00:30 covers 00:00-00:30 and the reading at 01:00 covers 00:30-01:00, so both belong to hour 00:00.

In [23]:
full["hour_start"] = (full["ts"] - pd.Timedelta(minutes=30)).dt.floor("h")
full.head()

,LCLid,ts,energy,hour_start
0,MAC003737,2013-01-01 00:00:00+00:00,0.088,2012-12-31 23:00:00+00:00
1,MAC003737,2013-01-01 00:30:00+00:00,0.109,2013-01-01 00:00:00+00:00
2,MAC003737,2013-01-01 01:00:00+00:00,0.075,2013-01-01 00:00:00+00:00
3,MAC003737,2013-01-01 01:30:00+00:00,0.076,2013-01-01 01:00:00+00:00
4,MAC003737,2013-01-01 02:00:00+00:00,0.108,2013-01-01 01:00:00+00:00


## Add readings for each hour

In [24]:
full_hourly = full.groupby(["LCLid", "hour_start"])["energy"].agg(n_valid="count", kWh_sum="sum")
full_hourly["kWh"] = full_hourly["kWh_sum"].where(full_hourly["n_valid"] == 2).round(3)
full_hourly.head()

n_valid  kWh_sum    kWh
LCLid     hour_start                                        
MAC000038 2012-12-31 23:00:00+00:00        1    0.484    NaN
          2013-01-01 00:00:00+00:00        2    1.188  1.188
          2013-01-01 01:00:00+00:00        2    0.503  0.503
          2013-01-01 02:00:00+00:00        2    0.164  0.164
          2013-01-01 03:00:00+00:00        2    0.143  0.143

## List all hours in 2013

Each home gets every hour from 2013-01-01 00:00 to 2013-12-31 23:00 UTC (8,760 hours).

In [25]:
full_hours = pd.date_range("2013-01-01 00:00", "2013-12-31 23:00", freq="h", tz="UTC")
full_grid = pd.MultiIndex.from_product([selected_homes, full_hours], names=["LCLid", "hour_start"])
len(full_hours), len(full_grid)

(8760, 2628000)

## Readings outside the 2013 hours

The first reading of each home ends at 2013-01-01 00:00, so it covers 2012-12-31 23:30-00:00 and belongs to hour
2012-12-31 23:00. That hour is outside 2013, so it is not part of the table.

In [26]:
outside = full[full["hour_start"] < full_hours[0]]
len(outside), outside["LCLid"].nunique(), outside["hour_start"].unique()

(300,
 300,
 <DatetimeArray>
 ['2012-12-31 23:00:00+00:00']
 Length: 1, dtype: datetime64[ns, UTC])

## Keep missing hours in the table

In [27]:
full_hourly = full_hourly.reindex(full_grid)
full_hourly["n_valid"] = full_hourly["n_valid"].fillna(0).astype(int)
full_hourly["is_complete"] = full_hourly["n_valid"] == 2
full_hourly = full_hourly[["kWh", "n_valid", "is_complete"]].reset_index()
full_hourly.head()

,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000038,2013-01-01 00:00:00+00:00,1.188,2,True
1,MAC000038,2013-01-01 01:00:00+00:00,0.503,2,True
2,MAC000038,2013-01-01 02:00:00+00:00,0.164,2,True
3,MAC000038,2013-01-01 03:00:00+00:00,0.143,2,True
4,MAC000038,2013-01-01 04:00:00+00:00,0.128,2,True


## Count total, complete and missing hours

In [28]:
total_hours = len(full_hourly)
complete_hours = int(full_hourly["is_complete"].sum())
missing_hours = int((~full_hourly["is_complete"]).sum())

hour_counts = {
    "homes": full_hourly["LCLid"].nunique(),
    "total_hours": total_hours,
    "complete_hours": complete_hours,
    "missing_hours": missing_hours,
    "missing_with_one_reading": int((full_hourly["n_valid"] == 1).sum()),
    "missing_with_no_reading": int((full_hourly["n_valid"] == 0).sum()),
    "repeated_home_hour": int(full_hourly.duplicated(["LCLid", "hour_start"]).sum()),
}
assert hour_counts["homes"] == 300
assert hour_counts["repeated_home_hour"] == 0
assert complete_hours + missing_hours == total_hours
assert full_hourly.loc[~full_hourly["is_complete"], "kWh"].isna().all()
hour_counts

{'homes': 300,
 'total_hours': 2628000,
 'complete_hours': 2609345,
 'missing_hours': 18655,
 'missing_with_one_reading': 826,
 'missing_with_no_reading': 17829,
 'repeated_home_hour': 0}

## Missing hours per home

Every home's last hour (2013-12-31 23:00) is missing: its second reading would end at 2014-01-01 00:00,
which is outside the raw file.

In [29]:
missing_per_home = (~full_hourly["is_complete"]).groupby(full_hourly["LCLid"]).sum()
last_hour = full_hourly[full_hourly["hour_start"] == full_hours[-1]]
print("homes with the last hour missing:", int((~last_hour["is_complete"]).sum()))
missing_per_home.describe()

homes with the last hour missing: 300


count     300.000000
mean       62.183333
std       304.548533
min         1.000000
25%         2.000000
50%         3.000000
75%         5.000000
max      2856.000000
Name: is_complete, dtype: float64

## First and last hours

In [30]:
full_hourly["hour_start"].min(), full_hourly["hour_start"].max()

(Timestamp('2013-01-01 00:00:00+0000', tz='UTC'),
 Timestamp('2013-12-31 23:00:00+0000', tz='UTC'))

## Five raw-pair examples

Five training hours (Jan-Aug 2013). For hour H, the two raw readings end at H+30 min and H+60 min.

In [31]:
pair_cases = [
    ("normal winter hour", "MAC001312", "2013-01-01 00:00"),
    ("home outside the sample", "MAC000038", "2013-01-02 01:00"),
    ("one reading is a measured zero", "MAC000675", "2013-01-01 10:00"),
    ("clock-change night, UTC", "MAC000414", "2013-03-31 01:00"),
    ("summer afternoon", "MAC000414", "2013-06-12 14:00"),
]
raw_lookup = full.set_index(["LCLid", "ts"])["energy"]
saved_lookup = full_hourly.set_index(["LCLid", "hour_start"])


def raw_pair_table(cases):
    table = pd.DataFrame(cases, columns=["case", "LCLid", "hour_start"])
    table["hour_start"] = pd.to_datetime(table["hour_start"], utc=True)
    table["first_end"] = table["hour_start"] + pd.Timedelta(minutes=30)
    table["second_end"] = table["hour_start"] + pd.Timedelta(hours=1)
    first_keys = pd.MultiIndex.from_arrays([table["LCLid"], table["first_end"]])
    second_keys = pd.MultiIndex.from_arrays([table["LCLid"], table["second_end"]])
    table["first_kwh"] = raw_lookup.reindex(first_keys).to_numpy()
    table["second_kwh"] = raw_lookup.reindex(second_keys).to_numpy()
    table["raw_sum"] = (table["first_kwh"] + table["second_kwh"]).round(3)
    saved_keys = pd.MultiIndex.from_arrays([table["LCLid"], table["hour_start"]])
    table["saved_kWh"] = saved_lookup["kWh"].reindex(saved_keys).to_numpy()
    table["saved_n_valid"] = saved_lookup["n_valid"].reindex(saved_keys).to_numpy()
    both_missing = table["raw_sum"].isna() & table["saved_kWh"].isna()
    table["matches"] = both_missing | ((table["raw_sum"] - table["saved_kWh"]).abs() < 1e-9)
    return table


pairs = raw_pair_table(pair_cases)
assert pairs["matches"].all() and pairs["saved_kWh"].notna().all()
show_columns = ["case", "LCLid", "hour_start", "first_end", "first_kwh",
                "second_end", "second_kwh", "raw_sum", "saved_kWh", "matches"]
pairs[show_columns]

,case,LCLid,hour_start,first_end,first_kwh,second_end,second_kwh,raw_sum,saved_kWh,matches
0,normal winter hour,MAC001312,2013-01-01 00:00:00+00:00,2013-01-01 00:30:00+00:00,0.018,2013-01-01 01:00:00+00:00,0.041,0.059,0.059,True
1,home outside the sample,MAC000038,2013-01-02 01:00:00+00:00,2013-01-02 01:30:00+00:00,0.056,2013-01-02 02:00:00+00:00,0.055,0.111,0.111,True
2,one reading is a measured zero,MAC000675,2013-01-01 10:00:00+00:00,2013-01-01 10:30:00+00:00,0.080,2013-01-01 11:00:00+00:00,0.000,0.080,0.080,True
3,"clock-change night, UTC",MAC000414,2013-03-31 01:00:00+00:00,2013-03-31 01:30:00+00:00,0.355,2013-03-31 02:00:00+00:00,0.317,0.672,0.672,True
4,summer afternoon,MAC000414,2013-06-12 14:00:00+00:00,2013-06-12 14:30:00+00:00,0.236,2013-06-12 15:00:00+00:00,0.182,0.418,0.418,True


## Hours with fewer than two readings stay missing

One hour with a single reading, one hour inside a day with no readings, and the last hour of 2013.

In [32]:
missing_cases = [
    ("only one reading", "MAC005456", "2013-01-26 08:00"),
    ("no readings that day", "MAC005456", "2013-08-18 05:00"),
    ("second reading would be in 2014", "MAC001312", "2013-12-31 23:00"),
]
missing_examples = raw_pair_table(missing_cases)
assert missing_examples["saved_kWh"].isna().all() and missing_examples["matches"].all()
missing_examples[show_columns + ["saved_n_valid"]]

,case,LCLid,hour_start,first_end,first_kwh,second_end,second_kwh,raw_sum,saved_kWh,matches,saved_n_valid
0,only one reading,MAC005456,2013-01-26 08:00:00+00:00,2013-01-26 08:30:00+00:00,NaN,2013-01-26 09:00:00+00:00,0.66,NaN,NaN,True,1
1,no readings that day,MAC005456,2013-08-18 05:00:00+00:00,2013-08-18 05:30:00+00:00,NaN,2013-08-18 06:00:00+00:00,NaN,NaN,NaN,True,0
2,second reading would be in 2014,MAC001312,2013-12-31 23:00:00+00:00,2013-12-31 23:30:00+00:00,0.027,2014-01-01 00:00:00+00:00,NaN,NaN,NaN,True,1


## Save the hourly file

In [33]:
processed_dir = PROJECT_ROOT / "data/processed"
processed_dir.mkdir(parents=True, exist_ok=True)
hourly_path = processed_dir / "hourly_readings.csv"
full_hourly.to_csv(hourly_path, index=False)
hourly_path

WindowsPath('C:/Projects/household-energy-prediction/data/processed/hourly_readings.csv')

## Open the saved file

Columns: `LCLid` (home ID), `hour_start` (UTC hour start), `kWh` (hourly energy, missing when incomplete),
`n_valid` (valid half-hour readings found, 0-2) and `is_complete` (True only when `n_valid` is 2).

In [34]:
saved = pd.read_csv(hourly_path)
saved["hour_start"] = pd.to_datetime(saved["hour_start"], utc=True)
print("rows:", len(saved), "| homes:", saved["LCLid"].nunique())
print("first hour:", saved["hour_start"].min(), "| last hour:", saved["hour_start"].max())
print(saved["is_complete"].value_counts())
pd.concat([saved.head(3), saved.tail(3)])

rows: 2628000 | homes: 300
first hour: 2013-01-01 00:00:00+00:00 | last hour: 2013-12-31 23:00:00+00:00
is_complete
True     2609345
False      18655
Name: count, dtype: int64


,LCLid,hour_start,kWh,n_valid,is_complete
0,MAC000038,2013-01-01 00:00:00+00:00,1.188,2,True
1,MAC000038,2013-01-01 01:00:00+00:00,0.503,2,True
2,MAC000038,2013-01-01 02:00:00+00:00,0.164,2,True
2627997,MAC005532,2013-12-31 21:00:00+00:00,0.258,2,True
2627998,MAC005532,2013-12-31 22:00:00+00:00,0.213,2,True
2627999,MAC005532,2013-12-31 23:00:00+00:00,NaN,1,False


# Part 3: Checks with M3 and M5 (M2-W2)

**Result:** no preparation error was found. Every checked value matches its original half-hour readings,
so `hourly_readings.csv` is unchanged and M3 and M5 do not need to rerun anything because of M2.

Part 3 uses the tables from Part 2 (`full`, `full_hourly`, `raw_pair_table`). Run Parts 1 and 2 first.

## Issue log

Each issue that M3 or M5 may see when reading the hourly file. The examples below check each one against the raw readings.

In [35]:
issue_log = pd.DataFrame([
    (1, "First 24 hours of 2013 have no lag values", "M3",
     "No 2012 data, so there is no previous hour or previous day for 1 January.", "Explained"),
    (2, "2013-12-31 23:00 is missing for every home", "M3, M4",
     "Its second reading would end at 2014-01-01 00:00, which is outside the raw file.", "Explained"),
    (3, "MAC000668 has many missing rows from September", "M3, M4",
     "Its raw readings stop on 2013-09-04; the raw file has nothing after that.", "Explained"),
    (4, "Very large hourly values (up to 11.062 kWh)", "M5",
     "Real raw readings; the home has many high half-hours, not one glitch.", "Explained"),
    (5, "Hours with exactly 0.000 kWh", "M5",
     "Both raw readings are 0.0. Measured zeros are valid, not missing.", "Explained"),
    (6, "Tiny differences when recomputing from raw", "M3",
     "Some raw values have extra digits (e.g. 5.5219998); hourly kWh is rounded to 3 decimals.", "Explained"),
    (7, "Daily profile peaks shifted by 1 hour in summer", "M5",
     "Hours are UTC (team rule). London is UTC+1 from 31 Mar to 27 Oct.", "Explained"),
], columns=["issue", "what_users_see", "affects", "explanation", "status"]).set_index("issue")
with pd.option_context("display.max_colwidth", None):
    display(issue_log)

,what_users_see,affects,explanation,status
issue,,,,
1,First 24 hours of 2013 have no lag values,M3,"No 2012 data, so there is no previous hour or previous day for 1 January.",Explained
2,2013-12-31 23:00 is missing for every home,"M3, M4","Its second reading would end at 2014-01-01 00:00, which is outside the raw file.",Explained
3,MAC000668 has many missing rows from September,"M3, M4",Its raw readings stop on 2013-09-04; the raw file has nothing after that.,Explained
4,Very large hourly values (up to 11.062 kWh),M5,"Real raw readings; the home has many high half-hours, not one glitch.",Explained
5,Hours with exactly 0.000 kWh,M5,"Both raw readings are 0.0. Measured zeros are valid, not missing.",Explained
6,Tiny differences when recomputing from raw,M3,Some raw values have extra digits (e.g. 5.5219998); hourly kWh is rounded to 3 decimals.,Explained
7,Daily profile peaks shifted by 1 hour in summer,M5,Hours are UTC (team rule). London is UTC+1 from 31 Mar to 27 Oct.,Explained


## M3 and M5 can read the same file

Read the saved file the way M3 and M5 do: keep the development homes and turn `hour_start` into UTC times.

In [36]:
split = pd.read_csv(PROJECT_ROOT / "data/household_split.csv")
development_homes = set(split.loc[split["Cohort"] == "DEVELOPMENT", "LCLid"])

users_view = pd.read_csv(hourly_path)
users_view["hour_start"] = pd.to_datetime(users_view["hour_start"], utc=True)
users_view = users_view[users_view["LCLid"].isin(development_homes)]
train_view = users_view[users_view["hour_start"] < pd.Timestamp("2013-09-01", tz="UTC")]

read_check = {
    "development_homes": users_view["LCLid"].nunique(),
    "development_rows": len(users_view),
    "one_row_per_home_hour": not users_view.duplicated(["LCLid", "hour_start"]).any(),
    "train_hours_with_kWh": int(train_view["kWh"].notna().sum()),
    "train_hours_missing": int(train_view["kWh"].isna().sum()),
}
assert read_check["development_homes"] == 240 and read_check["one_row_per_home_hour"]
read_check

{'development_homes': 240,
 'development_rows': 2102400,
 'one_row_per_home_hour': True,
 'train_hours_with_kWh': 1398898,
 'train_hours_missing': 782}

## Five checked training-hour totals

Five complete TRAIN hours linked to the issues above. Each saved total equals its two raw readings.

In [37]:
w2_cases = [
    ("issue 4: largest hourly value", "MAC004407", "2013-03-04 06:00"),
    ("issue 5: both readings are measured zeros", "MAC000675", "2013-01-01 09:00"),
    ("issue 1: first hour with a previous-day value", "MAC000038", "2013-01-02 00:00"),
    ("issue 7: summer evening (UTC)", "MAC000414", "2013-06-12 17:00"),
    ("issue 3: MAC000668 before its readings stop", "MAC000668", "2013-08-31 18:00"),
]
w2_pairs = raw_pair_table(w2_cases)
assert w2_pairs["matches"].all() and w2_pairs["saved_kWh"].notna().all()
assert (w2_pairs["hour_start"] < pd.Timestamp("2013-09-01", tz="UTC")).all()
w2_pairs[show_columns]

,case,LCLid,hour_start,first_end,first_kwh,second_end,second_kwh,raw_sum,saved_kWh,matches
0,issue 4: largest hourly value,MAC004407,2013-03-04 06:00:00+00:00,2013-03-04 06:30:00+00:00,5.540,2013-03-04 07:00:00+00:00,5.522,11.062,11.062,True
1,issue 5: both readings are measured zeros,MAC000675,2013-01-01 09:00:00+00:00,2013-01-01 09:30:00+00:00,0.000,2013-01-01 10:00:00+00:00,0.000,0.000,0.000,True
2,issue 1: first hour with a previous-day value,MAC000038,2013-01-02 00:00:00+00:00,2013-01-02 00:30:00+00:00,0.167,2013-01-02 01:00:00+00:00,0.060,0.227,0.227,True
3,issue 7: summer evening (UTC),MAC000414,2013-06-12 17:00:00+00:00,2013-06-12 17:30:00+00:00,0.280,2013-06-12 18:00:00+00:00,0.252,0.532,0.532,True
4,issue 3: MAC000668 before its readings stop,MAC000668,2013-08-31 18:00:00+00:00,2013-08-31 18:30:00+00:00,0.059,2013-08-31 19:00:00+00:00,1.151,1.210,1.210,True


## Issues 1, 2 and 3: hours that stay missing

- Issue 1: hour 2012-12-31 23:00 is not in the file, so 2013-01-01 00:00 has no previous-hour value.
- Issue 2: the last hour of 2013 has only one reading.
- Issue 3: MAC000668 has no readings after 2013-09-04 00:00.

In [38]:
day_one_lag = pd.Timestamp("2012-12-31 23:00", tz="UTC")
print("2012-12-31 23:00 in the hourly file:", bool((full_hourly["hour_start"] == day_one_lag).any()))
print("first hour in the hourly file:", full_hourly["hour_start"].min())

home_668 = full[full["LCLid"] == "MAC000668"]
print("MAC000668 last raw reading ends at:", home_668["ts"].max())
print("MAC000668 raw readings after that:", int((home_668["ts"] > home_668["ts"].max()).sum()))

w2_missing = raw_pair_table([
    ("issue 2: last hour of 2013", "MAC000038", "2013-12-31 23:00"),
    ("issue 3: last hour with MAC000668 readings", "MAC000668", "2013-09-03 23:00"),
    ("issue 3: first hour after MAC000668 stops", "MAC000668", "2013-09-04 00:00"),
])
assert w2_missing["matches"].all()
w2_missing[show_columns + ["saved_n_valid"]]

2012-12-31 23:00 in the hourly file: False
first hour in the hourly file: 2013-01-01 00:00:00+00:00
MAC000668 last raw reading ends at: 2013-09-04 00:00:00+00:00
MAC000668 raw readings after that: 0


,case,LCLid,hour_start,first_end,first_kwh,second_end,second_kwh,raw_sum,saved_kWh,matches,saved_n_valid
0,issue 2: last hour of 2013,MAC000038,2013-12-31 23:00:00+00:00,2013-12-31 23:30:00+00:00,0.395,2014-01-01 00:00:00+00:00,NaN,NaN,NaN,True,1
1,issue 3: last hour with MAC000668 readings,MAC000668,2013-09-03 23:00:00+00:00,2013-09-03 23:30:00+00:00,0.010,2013-09-04 00:00:00+00:00,0.027,0.037,0.037,True,2
2,issue 3: first hour after MAC000668 stops,MAC000668,2013-09-04 00:00:00+00:00,2013-09-04 00:30:00+00:00,NaN,2013-09-04 01:00:00+00:00,NaN,NaN,NaN,True,0


## Issue 3: missing hours for MAC000668 by month

In [39]:
home_668_hours = full_hourly[full_hourly["LCLid"] == "MAC000668"]
(~home_668_hours["is_complete"]).groupby(home_668_hours["hour_start"].dt.month).sum().rename("missing_hours")

hour_start
1       0
2       0
3       0
4       0
5       0
6       0
7       0
8       0
9     648
10    744
11    720
12    744
Name: missing_hours, dtype: int64

## Issue 4: the home with the largest values

In [40]:
home_4407 = full[full["LCLid"] == "MAC004407"]
{
    "half_hours_at_or_above_4_kWh": int((home_4407["energy"] >= 4).sum()),
    "mean_half_hour_kWh": round(home_4407["energy"].mean(), 3),
    "largest_half_hour_kWh_all_homes": full["energy"].max(),
    "largest_hourly_kWh_all_homes": full_hourly["kWh"].max(),
}

{'half_hours_at_or_above_4_kWh': 105,
 'mean_half_hour_kWh': 0.356,
 'largest_half_hour_kWh_all_homes': 7.2360001,
 'largest_hourly_kWh_all_homes': 11.062}

## Issue 5: measured zeros in the TRAIN development hours

In [41]:
int((train_view["kWh"] == 0).sum())

8430

## Issue 6: extra digits in the raw readings

Some raw values carry extra digits. The unrounded sum differs from the saved 3-decimal total by far less than
M3's 0.0005 tolerance.

In [42]:
raw_text = full_raw["energy(kWh/hh)"].str.strip()
extra_digits = raw_text.str.split(".").str[1].str.len() > 3
print("raw readings with more than 3 decimals:", int(extra_digits.sum()))
print("examples:", raw_text[extra_digits].head(3).tolist())

digits_check = w2_pairs.loc[[0], ["LCLid", "hour_start", "first_kwh", "second_kwh", "saved_kWh"]].copy()
digits_check["unrounded_sum"] = digits_check["first_kwh"] + digits_check["second_kwh"]
digits_check["difference"] = (digits_check["unrounded_sum"] - digits_check["saved_kWh"]).abs()
assert (digits_check["difference"] < 0.0005).all()
digits_check

raw readings with more than 3 decimals: 35655
examples: ['1.3430001', '1.0089999', '2.1489999']


,LCLid,hour_start,first_kwh,second_kwh,saved_kWh,unrounded_sum,difference
0,MAC004407,2013-03-04 06:00:00+00:00,5.54,5.522,11.062,11.062,2.000000e-07


## Issue 7: UTC hours and London clock time

In [43]:
utc_hours = pd.DatetimeIndex(["2013-01-15 17:00", "2013-06-12 17:00"], tz="UTC")
pd.DataFrame({"hour_start_utc": utc_hours, "london_clock": utc_hours.tz_convert("Europe/London")})

,hour_start_utc,london_clock
0,2013-01-15 17:00:00+00:00,2013-01-15 17:00:00+00:00
1,2013-06-12 17:00:00+00:00,2013-06-12 18:00:00+01:00


## What changed for M3 and M5

No preparation step changed. `hourly_readings.csv` is the same file as in Week 1, so M3 and M5 do not need to
rerun their notebooks because of M2.

## Replies from users

- **M3 (13 Nov):** all seven points match what M3 sees. M3's excluded rows (missing target 9,809,
  missing previous hour 713, missing previous day 7,146) come only from issues 1, 2 and 3:
  the first 24 hours of 2013, 2013-12-31 23:00 and MAC000668 after 2013-09-04. No other problems.
- **M5 (14 Nov):** ran the Week 2 charts and 24-hour daily profiles on hourly_readings.csv with no errors.
  Missing hours and measured zeros were handled as expected. No other problems with the hourly file.